<a href="https://colab.research.google.com/github/Ayush-0108/Indian-Bovine-Breed-Classifier/blob/main/cow_breed_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [52]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/

In [53]:
!kaggle datasets download lukex9442/indian-bovine-breeds


Dataset URL: https://www.kaggle.com/datasets/lukex9442/indian-bovine-breeds
License(s): CC0-1.0
indian-bovine-breeds.zip: Skipping, found more recently modified local copy (use --force to force download)


In [54]:
import tensorflow as tf
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense, Conv2D, MaxPooling2D, Flatten, BatchNormalization, Dropout

In [55]:
import zipfile
zip_ref = zipfile.ZipFile("/content/indian-bovine-breeds.zip","r")
zip_ref.extractall("/content")
zip_ref.close()

In [56]:
ds = keras.utils.image_dataset_from_directory(
    directory = "/content/Indian_bovine_breeds/Indian_bovine_breeds",
    labels = "inferred",
    label_mode = "int",
    batch_size = 32,
    image_size = (256,256),
    validation_split = 0.2,
    subset = "both",
    seed = 123
)

Found 5947 files belonging to 41 classes.
Using 4758 files for training.
Using 1189 files for validation.


The `ds` object is a tuple of two datasets: the training dataset and the validation dataset. Let's access the training dataset and see the shape of one batch of images and labels.

In [57]:
# The training dataset is ds[0]
train_ds = ds[0]

# Get the number of classes from the dataset
num_classes = len(train_ds.class_names)
print(f"Number of classes: {num_classes}")

# Iterate over the first batch of the training dataset
for images, labels in train_ds.take(1):
    print(f"Shape of images in the first batch: {images.shape}")
    print(f"Shape of labels in the first batch: {labels.shape}")
    print(f"Example labels: {labels.numpy()}")


Number of classes: 41
Shape of images in the first batch: (32, 256, 256, 3)
Shape of labels in the first batch: (32,)
Example labels: [ 9  7 34 21  3  6  9 33 39 18 32 10 12 17 33 11 40  4 10  9 34  6  9 14
 11 12 21  2 11  6 13 31]


In [58]:
test_ds = ds[1]

normalization


In [59]:
def process(image,label):
  image = tf.cast(image/255, tf.float32)
  return image,label
train_ds = train_ds.map(process)
test_ds = test_ds.map(process)

In [60]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

batch_size = 32
data_dir = "/content/Indian_bovine_breeds/Indian_bovine_breeds"

# Configure augmentation with a validation split
train_datagen = ImageDataGenerator(
    rescale=1./255,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2  # 20% of data will be reserved for validation
)

# We use the same generator object for validation to ensure identical rescaling
# but we won't apply augmentations to validation data usually.
# However, ImageDataGenerator applies the split to the original files.

train_generator = train_datagen.flow_from_directory(
    data_dir,
    target_size=(256, 256),
    batch_size=batch_size,
    class_mode='sparse',
    subset='training',  # Set as training data
    seed=123
)

validation_generator = train_datagen.flow_from_directory(
    data_dir,
    target_size=(256, 256),
    batch_size=batch_size,
    class_mode='sparse',
    subset='validation',  # Set as validation data
    seed=123
)

Found 4761 images belonging to 41 classes.
Found 1165 images belonging to 41 classes.


In [61]:
model = Sequential()
model.add(Conv2D(32,kernel_size = (3,3), padding = "valid",activation = "relu",input_shape = (256,256,3)))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size = (2,2),strides = 2, padding = "valid"))

model.add(Conv2D(64,kernel_size = (3,3), padding = "valid",activation = "relu"))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size = (2,2),strides = 2, padding = "valid"))

model.add(Conv2D(128,kernel_size = (3,3), padding = "valid",activation = "relu"))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size = (2,2),strides = 2, padding = "valid"))

model.add(Flatten())

model.add(Dense(128,activation = "relu"))
model.add(Dropout(0.1))
model.add(Dense(64,activation = "relu"))
model.add(Dropout(0.1))
model.add(Dense(41,activation = "softmax"))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [62]:
model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_12 (Conv2D)              │ (None, 254, 254, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_12          │ (None, 254, 254, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_12 (MaxPooling2D) │ (None, 127, 127, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_13 (Conv2D)              │ (None, 125, 125, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_13          │ (None, 125, 125, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_13 (MaxPooling2D) │ (None, 62, 62, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_14 (Conv2D)              │ (None, 60, 60, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_14          │ (None, 60, 60, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_14 (MaxPooling2D) │ (None, 30, 30, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_4 (Flatten)             │ (None, 115200)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 128)            │    14,745,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 41)             │         2,665 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,850,793 (56.65 MB)

 Trainable params: 14,850,345 (56.65 MB)

 Non-trainable params: 448 (1.75 KB)

In [63]:
model.compile(optimizer="adam",loss = "sparse_categorical_crossentropy", metrics = ["accuracy"])

In [ ]:
model.fit(
    train_generator,
    steps_per_epoch=100,
    epochs=5,
    validation_data=validation_generator,
    validation_steps=50
)

Epoch 1/5
 44/100 ━━━━━━━━━━━━━━━━━━━━ 53s 957ms/step - accuracy: 0.0413 - loss: 12.3992

### 1. Save the Trained Model

After training, you need to save your model's architecture, weights, and optimizer state. This allows you to load it later without rebuilding and retraining it.

In [ ]:
model.save('indian_bovine_breeds_model.keras') # Saves the model in the native Keras format
print("Model saved successfully!")

### 2. Create a Web Application (e.g., using Flask)

To turn your model into a website, you'll typically build a small web application. This application will:

*   Load your saved model.
*   Define an API endpoint (a URL) that receives image data.
*   Process the received image data (e.g., resize, normalize) to match what your model expects.
*   Use the loaded model to make a prediction.
*   Return the prediction (e.g., the breed name) as a response.

Here's a simplified example of how you might set up a Flask application. You would typically run this on a server, not directly in Colab.

In [ ]:
# This code block is for demonstration purposes and would typically run in a separate Python file for your web application.
# You would install Flask using: !pip install Flask

from flask import Flask, request, jsonify
import tensorflow as tf
from tensorflow import keras
import numpy as np
from PIL import Image
import io

app = Flask(__name__)

# Load the trained model
# Make sure the model path is correct relative to where your Flask app is running
loaded_model = keras.models.load_model('indian_bovine_breeds_model.keras')

# Assuming you have the class names available, e.g., from your dataset loading
# For this example, let's just create dummy class names or load them if available
# You might need to retrieve these from your `ds` object or a saved list.
num_classes = 41 # Based on your previous output
class_names = [f'Breed_{i}' for i in range(num_classes)] # Replace with actual breed names

@app.route('/predict', methods=['POST'])
def predict():
    if 'file' not in request.files:
        return jsonify({'error': 'No file part in the request'}), 400

    file = request.files['file']
    if file.filename == '':
        return jsonify({'error': 'No selected file'}), 400

    if file:
        try:
            # Read the image file
            img = Image.open(io.BytesIO(file.read()))
            # Resize and preprocess the image to match model input
            img = img.resize((256, 256))
            img_array = np.array(img)

            # Normalize as you did during training
            img_array = tf.cast(img_array / 255.0, tf.float32)
            # Add a batch dimension
            img_array = tf.expand_dims(img_array, 0) # (1, 256, 256, 3)

            # Make prediction
            predictions = loaded_model.predict(img_array)
            predicted_class_index = np.argmax(predictions[0])
            predicted_breed = class_names[predicted_class_index]

            return jsonify({'predicted_breed': predicted_breed, 'probabilities': predictions[0].tolist()})

        except Exception as e:
            return jsonify({'error': str(e)}), 500

# To run this Flask app:
# You would typically save this code as a Python file (e.g., `app.py`)
# and then run `python app.py` from your terminal on a server.
# If running locally for testing:
# if __name__ == '__main__':
#     app.run(debug=True) # debug=True is for development, set to False for production


### 3. Front-end (User Interface)

To make it a user-friendly website, you would also need a front-end interface (HTML, CSS, JavaScript). This front-end would allow users to:

*   Upload an image.
*   Send that image to your `/predict` API endpoint.
*   Display the prediction received from the API.

### 4. Deployment

Finally, you would deploy your Flask application (and optional front-end) to a cloud platform like Google Cloud Platform (e.g., App Engine, Cloud Run, Compute Engine), AWS, Heroku, etc. Each platform has its own set of steps for deployment.

In [ ]:
!pip install gradio

In [ ]:
import gradio as gr
import numpy as np
import tensorflow as tf
from tensorflow import keras

# Load the saved model
loaded_model = keras.models.load_model('indian_bovine_breeds_model.keras')

# Get the actual class names from the dataset object if available
# Since ds was created with image_dataset_from_directory, we can get class_names from it
real_class_names = ds[0].class_names

def predict_breed(img):
    # Preprocess image
    img_array = tf.image.resize(img, (256, 256))
    img_array = tf.cast(img_array / 255.0, tf.float32)
    img_array = tf.expand_dims(img_array, 0) # Create a batch

    # Predict
    prediction = loaded_model.predict(img_array).flatten()

    # Return a dictionary of class names and their probabilities for Gradio's Label component
    return {real_class_names[i]: float(prediction[i]) for i in range(len(real_class_names))}

# Create Gradio Interface
interface = gr.Interface(
    fn=predict_breed,
    inputs=gr.Image(),
    outputs=gr.Label(num_top_classes=3),
    title="Indian Bovine Breed Classifier",
    description="Upload an image of an Indian cow to identify its breed."
)

# Launch with share=True to get a public URL
interface.launch(share=True)

In [ ]:
print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))

If the output shows `Num GPUs Available: 0`, it means TensorFlow is not detecting a GPU. This could be due to:

1.  **No GPU:** Your Colab instance might be running on a CPU-only runtime.
2.  **Driver Issues:** GPU drivers might not be correctly installed or configured (less common in Colab, as it's usually pre-set).
3.  **TensorFlow Version:** An older TensorFlow version might not be compatible with the available GPU.

### How to enable GPU in Colab:

If you see `Num GPUs Available: 0`, you can change your Colab runtime type:

1.  Go to `Runtime` in the top menu.
2.  Select `Change runtime type`.
3.  In the `Hardware accelerator` dropdown, choose `GPU`.
4.  Click `Save`.

After changing the runtime, you will need to restart your Colab notebook and re-run all the cells from the beginning to ensure TensorFlow initializes with GPU support. If a GPU is detected (e.g., `Num GPUs Available: 1` or more), TensorFlow will usually leverage it automatically for operations like `model.fit()`.